# 03 Results (Phase 5, Gate 5)

Thin notebook over `src/evaluate.py`: every table below is produced by the same functions `python -m src.evaluate summary` uses. Run the CLI first (`evaluate run` per run, then `evaluate summary`); this notebook only reads `test_metrics.json` / `test_predictions.csv` and re-renders the tables and figures.

In [ ]:
# parameters: the smoke run sets RUNS_ROOT = 'data/processed/smoke_runs' and the smoke manifest dir
RUNS_ROOT = 'runs'
MANIFEST_DIR = None
FEATURE_ROOT = None
FUSION = 'fusion_gated'
UNIMODAL = ['text_roberta', 'audio_wavlm_ft']
OUT_DIR = None  # None -> reports/ (or data/processed/smoke_reports/ for smoke runs)

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
SMOKE = RUNS_ROOT != 'runs'
from src import evaluate as ev
OUT_DIR = Path(OUT_DIR) if OUT_DIR else (ROOT / ('data/processed/smoke_reports' if SMOKE else 'reports'))
runs_root = ROOT / RUNS_ROOT
df = ev.load_run_results(runs_root)
df[['name', 'seed', 'model_type', 'modality', 'text_source', 'n_test', 'weighted_f1', 'macro_f1', 'accuracy', 'fear_recall', 'disgust_recall', 'dev_weighted_f1', 'n_test_evaluations', 'timestamp']]

## Results ladder (V1, V2): mean +- std over seeds, 95 % bootstrap CIs

In [ ]:
ev.results_table(df)

## McNemar: fusion vs unimodal (V3)

In [ ]:
mc, best = ev.mcnemar_table(df, runs_root, FUSION, UNIMODAL)
print('best unimodal by dev weighted F1:', best)
mc

## Per-class F1 and confusion matrices (V4)

In [ ]:
display(ev.per_class_table(df))
for p in ev.confusion_figures(df, OUT_DIR / 'figures'):
    display(Markdown(f'**{p.name}**'))
    display(Image(filename=str(p), width=420))

## Transcript conditions (V5): train condition x test condition

In [ ]:
tt = ev.transcript_table(df)
if len(tt):
    for fam, g in tt.groupby('family'):
        display(Markdown(f'**{fam}** (weighted F1)'))
        display(g.pivot(index='train', columns='test', values='weighted_f1'))
else:
    print('no run evaluated with --text-conditions')

## Noise robustness (V6)

In [ ]:
ev.noise_table(df)

## Class-weight ablation (M7), dev gain (M8), train-dev gap (M9)

In [ ]:
display(ev.class_weight_table(df))
gg, info = ev.gain_gap_table(df, FUSION, UNIMODAL)
display(gg)
print(info)

## Literature comparison (V7)

In [ ]:
lit, filled = ev.literature_table(ROOT / 'reports/literature.yaml')
print(f'{filled} entries filled in (need 3)')
lit

## Error analysis (V8): 30 misclassified fusion test utterances, stratified by true class

In [ ]:
ea = ev.error_analysis(df, runs_root, FUSION, UNIMODAL, MANIFEST_DIR, OUT_DIR / 'error_analysis.csv', n=30)
if ea is not None:
    display(ea.groupby('true').agg(n=('utt_id', 'count'), mean_wer=('asr_wer', 'mean')))
    display(ea[['utt_id', 'true', 'pred', 'p_true', 'p_pred', 'asr_wer', 'modality_disagreement', 'gold_text', 'asr_text']])
else:
    print('no fusion test predictions')

## Gate 5 table (written by `evaluate summary`)

In [ ]:
p = OUT_DIR / 'gate5_eval.md'
display(Markdown(p.read_text()) if p.exists() else Markdown('_run `python -m src.evaluate summary` first_'))